# s14 - Evasion attack: evaluasi R5 (SURVEY)

**Klaim R5** (`paper-survey.md` §4.2): NIDS berbasis ML rapuh terhadap evasion
[F28][F33][F34]. Kami serang model global FL terlatih (dari R3) dengan serangan
yang **sudah ada** di literatur, pada **ruang fitur ter-standardisasi (z-space)**
dan **CONSTRAINED** (clip ke persentil valid) agar flow tetap plausibel (lihat
Lampiran B.5 soal kehati-hatian tabular-flow).

**Serangan diuji:** FGSM, PGD (iteratif), JSMA/saliency (pilih fitur paling
berpengaruh). Semua white-box pada MLP.

**Prasyarat:** `global_final.npz` dari run R3 ada di S3; partisi s01 ada.
Dilakukan **OFFLINE** (tak perlu EC2 — menyerang model yang sudah jadi).

**Output:** `survey_out/r5_evasion.csv` (ANGKA NYATA; jangan mengarang).

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='unsw-far/federated_survey'
RUN=os.environ.get('RUN','S_R3')
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)

## 1. Muat model global (R3) + data test; siapkan z-space & batas clip

In [ ]:
gf=load_npz_s3(f'{BASE}/run_{RUN}/global_final.npz')
assert gf is not None, 'PRASYARAT GAGAL: global_final.npz R3 belum ada (jalankan FedAvg EC2 + s03).'
W=[gf[f'w{i}'] for i in range(6)]
# data: pakai test gabungan + train gabungan utk z-score pemantauan (sama seperti s03)
P={}
for nm in ['cic_train','cic_test','unsw_train','unsw_test']:
    sub='cic' if nm.startswith('cic') else 'unsw'
    d=load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz'); P[nm]=(np.asarray(d['X'],float),np.asarray(d['y'],int))
Xtr=np.vstack([P['cic_train'][0],P['unsw_train'][0]])
Xte=np.vstack([P['cic_test'][0],P['unsw_test'][0]]); yte=np.concatenate([P['cic_test'][1],P['unsw_test'][1]])
zm=Xtr.mean(0); zs=Xtr.std(0); zs[zs==0]=1
Xz=(Xte-zm)/zs
# batas clip (persentil 1-99 z-space dari TRAIN) -> jaga plausibilitas flow
lo=np.percentile((Xtr-zm)/zs,1,axis=0); hi=np.percentile((Xtr-zm)/zs,99,axis=0)
print('test',Xz.shape,'pos',round(float(yte.mean()),3))

## 2. Forward + gradien input (white-box)

In [ ]:
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(a1,z1,a2,z2)
def dX_loss(W,X,y):
    o,(a1,z1,a2,z2)=fwd(W,X); g=(o-y.astype(float))[:,None]
    d2=(g@W[4].T)*(a2>0); d1=(d2@W[2].T)*(a1>0); return d1@W[0].T
def predict(W,X): o,_=fwd(W,X); return (o>=0.5).astype(int)
def mcc(W,X,y): return matthews_corrcoef(y,predict(W,X))

## 3. Serangan (semua CONSTRAINED: clip ke [lo,hi]). Target: flow ATTACK (y=1)

Evasion = buat serangan lolos sebagai benign. Kita ukur hanya pada sampel attack.

In [ ]:
def fgsm(W,X,y,eps): return np.clip(X+eps*np.sign(dX_loss(W,X,y)),lo,hi)
def pgd(W,X,y,eps,steps=10,alpha=None):
    alpha=alpha or eps/4; Xa=X.copy()
    for _ in range(steps):
        Xa=Xa+alpha*np.sign(dX_loss(W,Xa,y))
        Xa=np.clip(np.clip(Xa,X-eps,X+eps),lo,hi)  # dalam bola-eps DAN batas valid
    return Xa
def jsma(W,X,y,eps,k=2):
    # saliency: pilih k fitur dgn |dL/dX| terbesar per-sampel, perturbasi hanya itu
    d=dX_loss(W,X,y); Xa=X.copy()
    idxk=np.argsort(-np.abs(d),axis=1)[:,:k]
    for i in range(len(X)):
        j=idxk[i]; Xa[i,j]=np.clip(X[i,j]+eps*np.sign(d[i,j]),lo[j],hi[j])
    return Xa
atk_mask=(yte==1); Xa0=Xz[atk_mask]; ya0=yte[atk_mask]
base_recall=float((predict(W,Xa0)==1).mean())
print('baseline recall pada attack (clean):',round(base_recall,3))

## 4. Sweep epsilon untuk tiap serangan -> attack success rate + ΔMCC

In [ ]:
clean_mcc=mcc(W,Xz,yte)
def asr(Xadv):
    # attack success = sampel attack yang jadi diprediksi benign (0)
    return float((predict(W,Xadv)==0).mean())
def mcc_under(attack_fn,eps,**kw):
    Xz2=Xz.copy(); Xz2[atk_mask]=attack_fn(W,Xa0,ya0,eps,**kw)
    return mcc(W,Xz2,yte)
rows=[]
for eps in [0.05,0.1,0.2,0.5]:
    for nm,fn,kw in [('FGSM',fgsm,{}),('PGD',pgd,{}),('JSMA',jsma,{})]:
        Xadv=fn(W,Xa0,ya0,eps,**kw)
        rows.append(dict(attack=nm,eps=eps,asr=asr(Xadv),mcc_under=mcc_under(fn,eps,**kw),
                         clean_mcc=clean_mcc))
        print(f'{nm:5s} eps={eps:<4} ASR={rows[-1]["asr"]:.3f} MCC_under={rows[-1]["mcc_under"]:.3f} (clean {clean_mcc:.3f})')

## 5. Simpan R5 -> CSV + S3

In [ ]:
import csv
path=os.path.join(OUTDIR,'r5_evasion.csv')
with open(path,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['attack','eps','asr','mcc_under','clean_mcc']); w.writeheader()
    for r in rows: w.writerow(r)
print('saved',path)
try:
    s3.upload_file(path,BUCKET,f'{BASE}/results/r5_evasion.csv'); print('upload OK')
except Exception as e: print('(upload dilewati:',e,')')
print('=== s14 SELESAI. Isi §4.3 baris R5 dari CSV ini (ΔMCC = clean_mcc - mcc_under). ===')